# Model APIs, structured outputs, and typed decisions

Copyright (c) Patrick Hall (jphall@gwu.edu), 2026--27. Released under the MIT License.

This notebook sends customer complaints through the OpenAI Responses API, inspects request metadata, streams text, validates structured extractions with Pydantic, and evaluates bounded triage decisions.

## 1. Configure the OpenAI client
The client loads `OPENAI_API_KEY` from the repository-root `.env`; if it is unavailable, it requests the key without echoing it.

In [1]:
# 1. Configure the OpenAI client
# Locate the repository root so this notebook runs from the lecture directory or Jupyter's root.
import os
import sys
import time
from getpass import getpass
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from openai import APIConnectionError, APIStatusError, OpenAI, RateLimitError
from pydantic import BaseModel, Field, ValidationError, model_validator

ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'pyproject.toml').exists():
        ROOT = candidate
        break

LECTURE_DIR = ROOT / 'lecture' / 'lec_3'
# Put the lecture utility beside the notebook on the import path, exactly once.
if str(LECTURE_DIR) not in sys.path:
    sys.path.insert(0, str(LECTURE_DIR))

from utils import retry

load_dotenv(ROOT / '.env')
api_key = os.getenv('OPENAI_API_KEY') or getpass('OpenAI API key: ')
if not api_key:
    raise RuntimeError('An OpenAI API key is required to run this notebook.')

# A client-wide timeout and zero implicit retries keep retry behavior visible below.
client = OpenAI(api_key=api_key, timeout=45.0, max_retries=0)
MODEL = 'gpt-5-mini'
print(f'Configured {MODEL}; repository root: {ROOT.name}')

Configured gpt-5-mini; repository root: GWU_LLM


## 2. Create a bounded request
The complaint includes enough detail to extract a category, amount, and supporting text. Real applications should minimize sensitive data before calling an external service.

In [2]:
# 2. Create a bounded request
COMPLAINT = ('I was charged twice for $24.99 yesterday. The second charge is still pending. '
             'Please help me fix the duplicate charge.')
INSTRUCTIONS = ('You are a customer-support assistant. Use only the complaint text. '
                'If a fact is not stated, say that it is not stated. Keep the answer under 80 words.')
print(COMPLAINT)

I was charged twice for $24.99 yesterday. The second charge is still pending. Please help me fix the duplicate charge.


## 3. Send a request and inspect its complete response
Visible text is only one part of a response. The identifier, completion status, model, and token use are useful for diagnosis and cost accounting.

In [3]:
# 3. Send a request and inspect its complete response
response = client.responses.create(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=COMPLAINT,
    max_output_tokens=800,  # Leaves visible-output room after supported internal reasoning.
    reasoning={'effort': 'minimal'},
)

print('Text:', response.output_text)
print('Request ID:', response.id)
print('Status:', response.status)
print('Model:', response.model)
print('Usage:', response.usage)
if response.status != 'completed' or not response.output_text.strip():
    raise RuntimeError('The request did not produce completed visible text; inspect status and usage above.')

Text: You were charged twice for $24.99 yesterday. The second charge is still pending. You asked for help to fix the duplicate charge.
Request ID: resp_0ec70e963b2b113b006adf714182bc87d29339888b1ddef8f2
Status: completed
Model: gpt-5-mini-2025-08-07
Usage: ResponseUsage(input_tokens=71, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=46, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=117)


## 4. Compare supported operating settings
Holding the complaint constant lets us compare latency, output tokens, and content. Settings differ among models and providers, so this function records the actual model response rather than assuming a universal effect.

In [4]:
# 4. Compare supported operating settings
def run_configuration(label: str, reasoning_effort: str) -> dict:
    """Run one bounded configuration and retain comparable operational fields."""
    started = time.perf_counter()
    result = client.responses.create(
        model=MODEL, instructions=INSTRUCTIONS, input=COMPLAINT,
        max_output_tokens=800, reasoning={'effort': reasoning_effort},
    )
    return {
        'label': label,
        'seconds': round(time.perf_counter() - started, 2),
        'status': result.status,
        'output_tokens': getattr(result.usage, 'output_tokens', None),
        'text': result.output_text,
    }

comparison = [
    run_configuration('minimal reasoning', 'minimal'),
    run_configuration('medium reasoning', 'medium'),
]
for row in comparison:
    print(f"{row['label']}: {row['seconds']} s; {row['output_tokens']} output tokens; {row['status']}")
    print(' ', row['text'][:160].replace('\n', ' '))

minimal reasoning: 1.33 s; 46 output tokens; completed
  You were charged twice for $24.99 yesterday. The second charge is still pending. You ask for help to fix the duplicate charge.
medium reasoning: 6.27 s; 569 output tokens; completed
  I see you were charged twice for $24.99 yesterday and the second charge is still pending. To help, I need: your account/email or username; payment method or las


## 5. Stream a short acknowledgement
Streaming can improve perceived responsiveness for prose. A structured decision should be parsed only after a complete response is available.

In [5]:
# 5. Stream a short acknowledgement
stream = client.responses.create(
    model=MODEL,
    input='In one sentence, acknowledge a duplicate-charge complaint.',
    max_output_tokens=300,
    stream=True,
)
parts = []
for event in stream:
    if event.type == 'response.output_text.delta':
        parts.append(event.delta)
        print(event.delta, end='', flush=True)
        time.sleep(0.25) # add an exaggerated wait so streaming is more obvious
streamed_text = ''.join(parts)
print('\nComplete stream:', bool(streamed_text.strip()))

I'm sorry you've been charged twice; we acknowledge the duplicate charge and will investigate it immediately to correct the error and issue any necessary refund.
Complete stream: True


## 6. Define data contracts with Pydantic
Pydantic models both validate local Python data and generate JSON Schema for a structured-output request.

In [6]:
# 6. Define data contracts with Pydantic
class ComplaintFacts(BaseModel):
    category: Literal['billing', 'delivery', 'access']
    amount: float | None = Field(default=None, ge=0)
    evidence: str = Field(min_length=1)

class Refund(BaseModel):
    amount: float = Field(gt=0, le=250)
    currency: Literal['USD']

class TriageDecision(BaseModel):
    action: Literal['request_information', 'recommend_refund', 'escalate']
    rationale: str = Field(min_length=10)
    evidence: str = Field(min_length=1)
    question: str | None = None
    refund: Refund | None = None
    escalation_reason: str | None = None

    @model_validator(mode='after')
    def enforce_action_relationships(self):
        if self.action == 'recommend_refund' and self.refund is None:
            raise ValueError('A refund recommendation needs a refund object.')
        if self.action != 'recommend_refund' and self.refund is not None:
            raise ValueError('Only a refund recommendation may include a refund object.')
        if self.action == 'request_information' and not self.question:
            raise ValueError('A request for information needs a question.')
        if self.action == 'escalate' and not self.escalation_reason:
            raise ValueError('An escalation needs an escalation reason.')
        return self

print(ComplaintFacts.model_json_schema()['properties'].keys())

dict_keys(['category', 'amount', 'evidence'])


## 7. Extract structured complaint facts
`responses.parse` uses the Pydantic model to request a structured result, then exposes a locally parsed object. The evidence field remains a claim that should be reviewed against source text when accuracy matters.

In [7]:
# 7. Extract structured complaint facts
extraction = client.responses.parse(
    model=MODEL,
    instructions=('Extract only facts stated in the complaint. Use null for an unstated amount. '
                  'Choose exactly one category.'),
    input=COMPLAINT,
    text_format=ComplaintFacts,
    max_output_tokens=800,
    reasoning={'effort': 'minimal'},
)
facts = extraction.output_parsed
if extraction.status != 'completed' or facts is None:
    raise RuntimeError(f'Structured extraction failed with status: {extraction.status}')
print(facts.model_dump())
print('Schema-valid:', isinstance(facts, ComplaintFacts))

{'category': 'billing', 'amount': 24.99, 'evidence': 'I was charged twice for $24.99 yesterday. The second charge is still pending.'}
Schema-valid: True


## 8. Reject invalid local data
Schema-constrained generation reduces formatting failures, but applications must still reject malformed payloads received from any boundary or produced by later transformations.

In [8]:
# 8. Reject invalid local data
invalid_payload = {'category': 'billing', 'amount': -24.99}  # Missing evidence and invalid amount.
try:
    ComplaintFacts.model_validate(invalid_payload)
except ValidationError as error:
    for issue in error.errors():
        print(issue['loc'], '-', issue['msg'])

('amount',) - Input should be greater than or equal to 0
('evidence',) - Field required


## 9. Ask the Decisions API for a bounded choice
The Decisions API selects from explicitly supplied outcomes and returns a probability for every option plus a separate confidence value. The application still controls what any choice is allowed to do.

In [9]:
# 9. Ask the Decisions API for a bounded choice
DECISION_MODEL = 'gpt-6-luna'
POLICY = ('Refund a clear duplicate charge of $250 or less. '
          'Request more evidence when facts are missing. '
          'Escalate possible fraud or account takeover.')

decision = client.decisions.create(
    model=DECISION_MODEL,
    input=f'Policy: {POLICY}\nRecord: {COMPLAINT}',
    questions=[{
        'type': 'choice',
        'name': 'next_step',
        'instructions': 'Select the next step under the policy.',
        'choices': [
            {'value': 'refund', 'description': 'Recommend a refund'},
            {'value': 'abstain', 'description': 'Request more evidence'},
            {'value': 'escalate', 'description': 'Request human review'},
        ],
    }],
)
answer = decision.answers[0]
if answer.type == 'refusal':
    raise RuntimeError(f'Decision refused: {answer.name}')
if answer.type != 'choice':
    raise RuntimeError(f'Unexpected answer type: {answer.type}')

print('Selected next step:', answer.choice)
print(f'Confidence: {answer.confidence:.2f}')
print('Option probabilities:')
for option in answer.probabilities:
    print(f'  {option.value}: {option.probability:.2f}')
print('No refund is executed by this notebook; the choice only routes local review.')

Selected next step: abstain
Confidence: 0.76
Option probabilities:
  refund: 0.16
  abstain: 0.84
  escalate: 0.00
No refund is executed by this notebook; the choice only routes local review.


## 10. Use bounded retries for service failures
Transport problems and rate limits may be retried a small number of times. A refusal, incomplete response, or validation error needs a different path and should not be hidden by a retry loop.

In [10]:
# 10. Use bounded retries for service failures
def is_transient_service_error(error: Exception) -> bool:
    """Retry only connection, rate-limit, and temporary server failures."""
    return isinstance(error, (APIConnectionError, RateLimitError)) or (
        isinstance(error, APIStatusError) and error.status_code >= 500
    )

def report_retry(attempt: int, delay_seconds: float, error: Exception) -> None:
    print(f'Retrying after {delay_seconds:g} s because of {type(error).__name__} (attempt {attempt + 1}).')

@retry(
    max_attempts=3,
    base_delay_seconds=1.0,
    should_retry=is_transient_service_error,
    on_retry=report_retry,
)
def create_with_backoff(input_text: str):
    # The decorator applies the bounded retry policy around this single request.
    return client.responses.create(
        model=MODEL, input=input_text, max_output_tokens=800,
        reasoning={'effort': 'minimal'},
    )

print('Retry-decorated request function defined. Do not manufacture an outage merely to test retries.')

Retry-decorated request function defined. Do not manufacture an outage merely to test retries.


## 11. Evaluate structural, policy, and substantive behavior
Each labeled case makes a bounded Decisions API call. Check the returned choice and probability distribution, a deterministic local policy rule, and label correctness separately.

In [11]:
# 11. Evaluate structural, policy, and substantive behavior
NEXT_STEP_CHOICES = [
    {'value': 'refund', 'description': 'Recommend a refund'},
    {'value': 'abstain', 'description': 'Request more evidence'},
    {'value': 'escalate', 'description': 'Request human review'},
]
NEXT_STEP_VALUES = {choice['value'] for choice in NEXT_STEP_CHOICES}

EVALUATION_CASES = [
    {'id': 'duplicate_charge', 'text': 'I was charged twice for $24.99 yesterday.',
     'action': 'refund'},
    {'id': 'possible_takeover', 'text': 'A charge appeared after my email and shipping address changed.',
     'action': 'escalate'},
    {'id': 'missing_amount', 'text': 'I think I was charged twice, but I cannot see the amount.',
     'action': 'abstain'},
]

def policy_allows(case: dict, action: str) -> bool:
    # Deterministic local logic checks only the small policy encoded for these examples.
    text = case['text'].lower()
    duplicate = 'charged twice' in text
    has_amount = '$' in text
    takeover_signal = 'email' in text or 'shipping address' in text
    if action == 'refund':
        return duplicate and has_amount and not takeover_signal
    if action == 'abstain':
        return duplicate and not has_amount and not takeover_signal
    return action == 'escalate' and takeover_signal

def evaluate_case(case: dict) -> dict:
    # One Decisions API call per case makes the evaluation O(n) network requests.
    result = client.decisions.create(
        model=DECISION_MODEL,
        input=f'Policy: {POLICY}\nRecord: {case["text"]}',
        questions=[{
            'type': 'choice',
            'name': 'next_step',
            'instructions': 'Select the next step under the policy.',
            'choices': NEXT_STEP_CHOICES,
        }],
    )
    answer = result.answers[0]
    if answer.type != 'choice':
        return {'id': case['id'], 'choice_valid': False, 'policy_valid': False,
                'action_correct': False, 'predicted_action': None, 'probabilities': None}

    probabilities = {item.value: item.probability for item in answer.probabilities}
    choice_valid = (
        answer.choice in NEXT_STEP_VALUES
        and set(probabilities) == NEXT_STEP_VALUES
        and abs(sum(probabilities.values()) - 1.0) < 1e-6
    )
    return {
        'id': case['id'],
        'choice_valid': choice_valid,
        'policy_valid': choice_valid and policy_allows(case, answer.choice),
        'action_correct': answer.choice == case['action'],
        'predicted_action': answer.choice,
        'probabilities': probabilities,
    }

# This list comprehension preserves case-level results while scaling linearly with case count.
evaluation = [evaluate_case(case) for case in EVALUATION_CASES]
for row in evaluation:
    print(row)
print('Choice-valid rate:', sum(r['choice_valid'] for r in evaluation) / len(evaluation))
print('Policy-valid rate:', sum(r['policy_valid'] for r in evaluation) / len(evaluation))
print('Action accuracy:', sum(r['action_correct'] for r in evaluation) / len(evaluation))

{'id': 'duplicate_charge', 'choice_valid': True, 'policy_valid': True, 'action_correct': True, 'predicted_action': 'refund', 'probabilities': {'refund': 0.63, 'abstain': 0.36, 'escalate': 0.01}}
{'id': 'possible_takeover', 'choice_valid': True, 'policy_valid': False, 'action_correct': False, 'predicted_action': 'abstain', 'probabilities': {'refund': 0.0, 'abstain': 0.62, 'escalate': 0.38}}
{'id': 'missing_amount', 'choice_valid': True, 'policy_valid': True, 'action_correct': True, 'predicted_action': 'abstain', 'probabilities': {'refund': 0.0, 'abstain': 1.0, 'escalate': 0.0}}
Choice-valid rate: 1.0
Policy-valid rate: 0.6666666666666666
Action accuracy: 0.6666666666666666


A dependable LLM application bounds requests, observes response metadata, separates service failures from validation failures, constrains data with schemas, enforces business rules locally, makes abstention and escalation explicit, and evaluates structural validity separately from substantive correctness. Lecture 7 expands the evaluation and validation methods.